# 00b — Data Augmentation: Cobertura completa de selecciones

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)
**Fase:** 0.5 — Integración de fuentes complementarias

## Objetivo

El EDA de StatsBomb (notebook `00_eda.ipynb`) reveló un sesgo estructural: cobertura excelente para Europa, deficiente para CONMEBOL / CAF / AFC / CONCACAF. Para predecir el **Mundial 2026 (48 selecciones)** necesitamos cobertura completa.

Este notebook integra dos fuentes complementarias **gratuitas**:

1. **International football results** (`martj42/international_results`) — resultados de TODOS los partidos internacionales desde 1872. Cobertura total de selecciones.
2. **ELO histórico computado desde cero** — usando la fórmula del *World Football Elo Ratings* directamente sobre el dataset anterior. Reproducible, sin scraping.

## Por qué calcular ELO nosotros mismos

En vez de scrapear `eloratings.net`, computamos el ELO en código:

- **Reproducibilidad:** dado el dataset, cualquiera puede correr el notebook y obtener exactamente los mismos ratings.
- **Control de hiperparámetros:** K-factor, ventaja local, ajuste por goal difference — todo explícito.
- **Time-series correcto:** podemos extraer el ELO de cada equipo *en cualquier fecha del pasado*, evitando data leakage en entrenamiento.

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import os, sys, json
from pathlib import Path
from collections import defaultdict
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_RAW = Path(PROJECT_ROOT) / 'data' / 'raw'
DATA_INTERIM = Path(PROJECT_ROOT) / 'data' / 'interim'
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

INT_RESULTS_DIR = DATA_RAW / 'international_results'
INT_RESULTS_DIR.mkdir(exist_ok=True)

print(f'DATA_RAW = {DATA_RAW}')
print(f'DATA_INTERIM = {DATA_INTERIM}')

---
## 2. Descargar International Results

Usamos el GitHub mirror del dataset Kaggle — evita el setup de credenciales de Kaggle CLI.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/martj42/international_results/master'
FILES = ['results.csv', 'goalscorers.csv', 'shootouts.csv', 'former_names.csv']

for filename in FILES:
    target = INT_RESULTS_DIR / filename
    if target.exists():
        print(f'✓ {filename} already exists ({target.stat().st_size / 1024:.1f} KB)')
        continue
    print(f'→ Downloading {filename}...')
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{filename}', target)
        print(f'  ✓ Saved to {target} ({target.stat().st_size / 1024:.1f} KB)')
    except Exception as e:
        print(f'  ✗ Error: {e}')

In [ ]:
# Cargar el dataset principal
df_int = pd.read_csv(INT_RESULTS_DIR / 'results.csv', parse_dates=['date'])
print(f'Total partidos internacionales (desde {df_int.date.min().date()}): {len(df_int):,}')
print(f'\nColumnas: {df_int.columns.tolist()}')
df_int.head()

In [ ]:
# Filtrar a la era moderna (decisión Fase 0)
MIN_DATE = pd.Timestamp('2010-01-01')
df_modern = df_int[df_int.date >= MIN_DATE].copy().reset_index(drop=True)
print(f'Partidos post-2010: {len(df_modern):,}')
print(f'Selecciones únicas: {len(set(df_modern.home_team) | set(df_modern.away_team)):,}')

# Distribución por torneo
df_modern.tournament.value_counts().head(20).to_frame('count')

In [ ]:
# Mapeo de tournament → tournament_class (para K-factor del ELO)
TOURNAMENT_CLASS_MAP = {
    'FIFA World Cup': 'world_cup_final',
    'FIFA World Cup qualification': 'wc_qualifier',
    'UEFA Euro': 'continental_championship',
    'UEFA Euro qualification': 'continental_qualifier',
    'Copa América': 'continental_championship',
    'African Cup of Nations': 'continental_championship',
    'African Cup of Nations qualification': 'continental_qualifier',
    'AFC Asian Cup': 'continental_championship',
    'AFC Asian Cup qualification': 'continental_qualifier',
    'CONCACAF Gold Cup': 'continental_championship',
    'CONCACAF Nations League': 'continental_qualifier',
    'UEFA Nations League': 'continental_qualifier',
    'Confederations Cup': 'major',
    'Friendly': 'friendly',
}

df_modern['tournament_class'] = df_modern.tournament.map(TOURNAMENT_CLASS_MAP).fillna('other')
df_modern.tournament_class.value_counts()

---
## 3. Cómputo de ratings ELO desde cero

### Fórmula (World Football Elo Ratings)

Update post-partido para el equipo A:

$$R'_A = R_A + K \cdot G \cdot (S_A - E_A)$$

Donde:
- $S_A \in \{1, 0.5, 0\}$ — resultado real (W/D/L)
- $E_A = \dfrac{1}{1 + 10^{(R_B - R_A - H)/400}}$ — resultado esperado
- $H = 100$ si A juega de local, 0 si neutral, $-100$ si visitante
- $K$ — importancia del torneo (definido abajo)
- $G$ — multiplicador por goal difference:
  - $G = 1$ si diff $\leq 1$
  - $G = 1.5$ si diff $= 2$
  - $G = (11 + |\text{diff}|)/8$ si diff $\geq 3$

Todos los equipos arrancan en 1500 (rating ELO estándar).

In [ ]:
K_FACTORS = {
    'world_cup_final': 60,
    'continental_championship': 50,
    'wc_qualifier': 40,
    'continental_qualifier': 30,
    'major': 50,
    'other': 30,
    'friendly': 20,
}

INITIAL_ELO = 1500
HOME_ADVANTAGE = 100

def goal_diff_multiplier(goal_diff):
    gd = abs(goal_diff)
    if gd <= 1:
        return 1.0
    elif gd == 2:
        return 1.5
    else:
        return (11 + gd) / 8

def expected_score(rating_a, rating_b, home_adv):
    diff = rating_b - rating_a - home_adv
    return 1.0 / (1.0 + 10.0 ** (diff / 400.0))

In [ ]:
def compute_elo_history(df, initial_elo=INITIAL_ELO):
    """Run ELO sequentially over the full match history.
    Returns:
      - elo_history: list of dicts {date, team, elo_before, elo_after, opponent, ...}
      - final_elos: dict {team -> ELO at end of dataset}
    """
    elos = defaultdict(lambda: initial_elo)
    history = []
    
    # CRÍTICO: ordenar cronológicamente. Sin orden, el ELO carece de sentido.
    df_sorted = df.sort_values('date').reset_index(drop=True)
    
    for _, row in tqdm(df_sorted.iterrows(), total=len(df_sorted), desc='Computing ELO'):
        home, away = row.home_team, row.away_team
        hs, as_ = row.home_score, row.away_score
        if pd.isna(hs) or pd.isna(as_):
            continue
        
        # Determinar ventaja local
        home_adv = 0 if row.neutral else HOME_ADVANTAGE
        
        # Resultado real
        if hs > as_:
            s_home, s_away = 1.0, 0.0
        elif hs < as_:
            s_home, s_away = 0.0, 1.0
        else:
            s_home = s_away = 0.5
        
        # K-factor
        K = K_FACTORS.get(row.tournament_class, 30)
        
        # Goal diff multiplier
        G = goal_diff_multiplier(hs - as_)
        
        # Expected scores
        elo_home_before = elos[home]
        elo_away_before = elos[away]
        e_home = expected_score(elo_home_before, elo_away_before, home_adv)
        e_away = 1.0 - e_home  # zero-sum
        
        # Updates
        elo_home_after = elo_home_before + K * G * (s_home - e_home)
        elo_away_after = elo_away_before + K * G * (s_away - e_away)
        
        elos[home] = elo_home_after
        elos[away] = elo_away_after
        
        history.append({
            'date': row.date,
            'home_team': home,
            'away_team': away,
            'home_score': hs,
            'away_score': as_,
            'tournament': row.tournament,
            'tournament_class': row.tournament_class,
            'neutral': row.neutral,
            'home_elo_before': elo_home_before,
            'away_elo_before': elo_away_before,
            'home_elo_after': elo_home_after,
            'away_elo_after': elo_away_after,
            'expected_home_win_prob': e_home,
            'K_factor': K,
            'G_multiplier': G,
        })
    
    return pd.DataFrame(history), dict(elos)

# Importante: usamos TODA la historia (no solo post-2010) para que el ELO al inicio de 2010 ya esté "caliente"
# Aplicamos también el mapeo de tournament_class al dataset completo
df_int['tournament_class'] = df_int.tournament.map(TOURNAMENT_CLASS_MAP).fillna('other')

elo_history, final_elos = compute_elo_history(df_int)
print(f'\nELO history computed for {len(elo_history):,} matches')
print(f'Final ratings for {len(final_elos)} teams')

In [ ]:
# Sanity check: top 20 selecciones por ELO al final del dataset
elo_ranking = pd.Series(final_elos).sort_values(ascending=False)
print('Top 20 selecciones por ELO final:')
print(elo_ranking.head(20).to_frame('elo').round(1))

# Expected: Brazil, Argentina, France, Spain, Portugal, England, Belgium, Germany cerca del top
# Si el ranking se ve raro, hay bug en el cómputo

In [ ]:
# Visualizar la evolución del ELO de algunas selecciones top
import matplotlib.dates as mdates

teams_to_plot = ['Brazil', 'Argentina', 'France', 'Germany', 'Spain', 'Portugal', 'England', 'Belgium', 'Croatia', 'Netherlands']

fig, ax = plt.subplots(figsize=(16, 8))
for team in teams_to_plot:
    team_matches = elo_history[(elo_history.home_team == team) | (elo_history.away_team == team)].copy()
    team_matches['team_elo'] = np.where(team_matches.home_team == team,
                                         team_matches.home_elo_after,
                                         team_matches.away_elo_after)
    team_recent = team_matches[team_matches.date >= '2010-01-01']
    ax.plot(team_recent.date, team_recent.team_elo, label=team, alpha=0.8, linewidth=1.5)

ax.set_title('Evolución del ELO — Top selecciones (2010–2025)')
ax.set_xlabel('Fecha')
ax.set_ylabel('Rating ELO')
ax.legend(ncol=5, loc='lower right')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
plt.show()

**Sanity check visual:** esperamos picos de Alemania en 2014 (WC), de Francia en 2018 y 2022, de Argentina en 2022. Caídas notables de España post-2014, Italia post-2016, etc.

---
## 4. Análisis de cobertura por selección

Para cada selección candidata del Mundial 2026: ¿cuántos partidos tenemos en el período relevante (2014–presente)?

In [ ]:
# Lista de selecciones probablemente clasificadas o jugando repechaje para WC 2026
# (basado en clasificación al 16-may-2026; algunas pueden cambiar)
WC_2026_LIKELY_TEAMS = [
    # Anfitriones
    'United States', 'Mexico', 'Canada',
    # CONMEBOL
    'Argentina', 'Brazil', 'Uruguay', 'Colombia', 'Ecuador', 'Paraguay', 'Peru', 'Venezuela',
    # UEFA - top tier
    'France', 'Germany', 'Spain', 'England', 'Italy', 'Portugal', 'Netherlands', 'Belgium',
    'Croatia', 'Switzerland', 'Denmark', 'Poland', 'Austria', 'Hungary', 'Sweden', 'Norway',
    'Czech Republic', 'Serbia', 'Ukraine', 'Turkey', 'Romania',
    # CAF
    'Morocco', 'Senegal', 'Egypt', 'Algeria', 'Nigeria', 'Cameroon', 'Tunisia',
    'Ivory Coast', 'Ghana', 'South Africa',
    # AFC
    'Japan', 'South Korea', 'Iran', 'Saudi Arabia', 'Australia', 'Qatar', 'Iraq', 'Uzbekistan',
    # CONCACAF (additional)
    'Costa Rica', 'Panama', 'Jamaica', 'Honduras',
    # OFC
    'New Zealand',
]

print(f'Total selecciones a analizar: {len(WC_2026_LIKELY_TEAMS)}')

In [ ]:
# Métricas de cobertura para cada selección candidata
coverage_records = []
df_for_coverage = df_modern[df_modern.date >= '2014-01-01']  # 12 años hasta hoy

for team in WC_2026_LIKELY_TEAMS:
    team_matches = df_for_coverage[
        (df_for_coverage.home_team == team) | (df_for_coverage.away_team == team)
    ]
    
    # Cuántos partidos por categoría
    n_total = len(team_matches)
    n_wc = len(team_matches[team_matches.tournament == 'FIFA World Cup'])
    n_wc_qual = len(team_matches[team_matches.tournament == 'FIFA World Cup qualification'])
    n_friendly = len(team_matches[team_matches.tournament == 'Friendly'])
    n_continental = len(team_matches[team_matches.tournament_class.isin(['continental_championship', 'continental_qualifier'])])
    
    # ELO actual
    current_elo = final_elos.get(team, np.nan)
    
    coverage_records.append({
        'team': team,
        'elo': current_elo,
        'total_matches_2014plus': n_total,
        'wc_matches': n_wc,
        'wc_qualifier_matches': n_wc_qual,
        'continental_matches': n_continental,
        'friendly_matches': n_friendly,
    })

df_coverage = pd.DataFrame(coverage_records).sort_values('elo', ascending=False).reset_index(drop=True)
df_coverage.round(1).head(60)

In [ ]:
# Visualización: cobertura por confederación
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Histograma de partidos totales por equipo
axes[0].barh(df_coverage.team, df_coverage.total_matches_2014plus, color='steelblue')
axes[0].set_xlabel('# partidos internacionales (2014–presente)')
axes[0].set_title('Cobertura por selección — partidos internacionales')
axes[0].invert_yaxis()
axes[0].tick_params(axis='y', labelsize=7)

# ELO por equipo
df_coverage_sorted = df_coverage.sort_values('elo', ascending=True)
axes[1].barh(df_coverage_sorted.team, df_coverage_sorted.elo, color='firebrick')
axes[1].set_xlabel('ELO rating')
axes[1].set_title('ELO actual de cada selección candidata')
axes[1].tick_params(axis='y', labelsize=7)
axes[1].axvline(1500, color='gray', linestyle='--', alpha=0.5, label='ELO inicial (1500)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Resumen
print(f'\nCobertura mínima: {df_coverage.total_matches_2014plus.min()} partidos ({df_coverage.loc[df_coverage.total_matches_2014plus.idxmin(), "team"]})')
print(f'Cobertura máxima: {df_coverage.total_matches_2014plus.max()} partidos ({df_coverage.loc[df_coverage.total_matches_2014plus.idxmax(), "team"]})')
print(f'Cobertura mediana: {df_coverage.total_matches_2014plus.median():.0f} partidos')

**Hallazgo esperado:** todas las selecciones candidatas a WC 2026 deberían tener al menos 50+ partidos internacionales desde 2014. Si alguna tiene <30, marcarla como "low-coverage" — el modelo va a tener que tratarla con más cautela.

---
## 5. Cruce con StatsBomb: ¿qué equipos tienen también datos de eventos?

Para el Transformer (Fase 3+) queremos saber qué selecciones tienen lineups detallados disponibles.

In [ ]:
STATSBOMB_PATH = DATA_RAW / 'statsbomb' / 'data'

if STATSBOMB_PATH.exists():
    with open(STATSBOMB_PATH / 'competitions.json') as f:
        sb_competitions = pd.DataFrame(json.load(f))
    
    # Cargar todos los matches de StatsBomb para extraer los equipos
    sb_teams = set()
    for _, row in sb_competitions.iterrows():
        matches_path = STATSBOMB_PATH / 'matches' / str(row.competition_id) / f'{row.season_id}.json'
        if not matches_path.exists():
            continue
        with open(matches_path) as f:
            matches = json.load(f)
        for m in matches:
            sb_teams.add(m['home_team']['home_team_name'])
            sb_teams.add(m['away_team']['away_team_name'])
    
    # ¿Cuántas selecciones del WC2026 tienen al menos algún partido en StatsBomb?
    # OJO: StatsBomb usa nombres a veces distintos al dataset internacional
    in_statsbomb = []
    for team in WC_2026_LIKELY_TEAMS:
        # Match exacto + variantes comunes
        candidates = [team, team.replace('United States', 'USA'), team.replace('South Korea', 'Korea Republic'),
                      team.replace('Czech Republic', 'Czechia'), team.replace('Ivory Coast', "Côte d'Ivoire")]
        has_data = any(c in sb_teams for c in candidates)
        in_statsbomb.append(has_data)
    
    df_coverage['has_statsbomb_data'] = in_statsbomb
    
    n_with = df_coverage.has_statsbomb_data.sum()
    print(f'Selecciones candidatas con datos de StatsBomb: {n_with} / {len(WC_2026_LIKELY_TEAMS)}')
    print(f'\nSin datos de StatsBomb (van a depender solo de features tabulares):')
    print(df_coverage[~df_coverage.has_statsbomb_data].team.tolist())
else:
    print('StatsBomb data no encontrada. Saltando este análisis.')
    df_coverage['has_statsbomb_data'] = False

---
## 6. Persistir los datasets procesados

Guardamos todo en formato Parquet (compresión + acceso rápido) para uso en Fase 1.

In [ ]:
# 1. Dataset principal: resultados + ELO pre/post para cada partido
out_path = DATA_INTERIM / 'international_matches_with_elo.parquet'
elo_history.to_parquet(out_path, compression='snappy')
print(f'✓ Saved {len(elo_history):,} matches → {out_path}')
print(f'  Size: {out_path.stat().st_size / 1024 / 1024:.2f} MB')

# 2. Tabla de cobertura por selección (para reference)
out_path = DATA_INTERIM / 'wc2026_team_coverage.parquet'
df_coverage.to_parquet(out_path, compression='snappy')
print(f'✓ Saved coverage table → {out_path}')

# 3. ELO snapshot final (para inicializar baselines)
out_path = DATA_INTERIM / 'elo_final_ratings.parquet'
elo_ranking.to_frame('elo').reset_index().rename(columns={'index': 'team'}).to_parquet(out_path)
print(f'✓ Saved {len(elo_ranking)} team ratings → {out_path}')

---
## 7. Conclusiones de Fase 0.5

Llenar al final de la ejecución:

- [ ] Total partidos internacionales (post-2010): ___
- [ ] # selecciones únicas en el dataset: ___
- [ ] Cobertura mínima para selecciones WC 2026: ___ partidos
- [ ] Selecciones WC 2026 con datos StatsBomb: ___ / ___
- [ ] Top 5 ELO al cierre del dataset: ___
- [ ] ¿La evolución del ELO se ve plausible? (Picos en 2014/2018/2022 para campeones) ___

## Verificación de comprensión (Fase 0.5)

1. ¿Por qué es CRÍTICO ordenar el dataset cronológicamente antes de computar el ELO? ¿Qué tipo de "leakage temporal" estaríamos introduciendo si lo computáramos en otro orden?
2. El multiplicador $G$ por diferencia de goles transforma una victoria 5-0 en un evento que mueve el ELO mucho más que una victoria 1-0. Esta decisión de diseño asume implícitamente algo sobre la **señal informativa de un blowout vs. una victoria ajustada**. ¿Cuál es esa asunción? ¿Puede ser problemática en algún contexto?